# pinn_impact_chain.ipynb

Comprehensive PINN workflow for a **20-DOF free-free chain** with:
- **no external force**,
- **left-end initial velocity excitation only**,
- sequential simulation up to **10s**.

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from pinn_impact_chain_solver import (
    PIPNNs,
    build_free_free_chain_matrices,
    make_left_velocity_ic,
    find_impact_times,
    propagate_ics,
    newmark_beta,
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
np.random.seed(1234)
tf.random.set_seed(1234)
print('TF version:', tf.__version__)


## 1) Parameters


In [ ]:
# --- physical system ---
n_dof = 20
m_x = 1.0
m_y = 0.3
k = 1.0
c = 0.0
D = 1.0
r = 1.0

# no external forcing in this study
phi1 = 0.0
phi2 = 0.0

# at least 3 input velocities/energies (matching low/medium/high style)
input_velocity_cases = {
    'low': -1.0,
    'medium': -2.0,
    'high': -10.0,
}

# default IC used by the single-case Newmark sanity plot (batch cases are run later)
x1_0 = 0.0
v1_0 = input_velocity_cases['medium']

# --- PINN/training controls ---
layers = [1, 64, n_dof]
hyp_ini_weight_loss = np.array([1.0, 1.0])
optimizer_LB_value = True

# per-segment horizon for impact search
T_segment = 1.0
n_t_segment = 100
nIter_per_segment = 1000

# total physical time to simulate for each case
t_end_target = 10.0

# contact-time assumption to convert impulse J -> equivalent avg impact force F=J/tau
tau_contact = 1e-3

# optional Newmark reference for early-time sanity check
T_ref = 10.0
dt_ref = 0.001

## 2) Build free-free matrices and initial conditions


In [ ]:
M_total, C_total, K_total = build_free_free_chain_matrices(
    n_dof=n_dof, m_x=m_x, k=k, c=c
)

x0_total, xt0_total = make_left_velocity_ic(
    n_dof=n_dof, x1_0=x1_0, v1_0=v1_0
)

y0 = np.zeros(n_dof)
yt0 = np.zeros(n_dof)

# impact mapping matrix (same collision law structure)
A = np.array([[m_x, m_y],
              [1.0, -1.0]])
B = np.array([[m_x, m_y],
              [-r,   r  ]])
A_inv_B = np.linalg.inv(A) @ B

print('M/K/C shape:', M_total.shape, K_total.shape, C_total.shape)
print('Initial x1, v1:', x0_total[0, 0], xt0_total[0, 0])


## 3) Newmark early-time reference (no impactors, no external force)


In [ ]:
num_ref = int(T_ref / dt_ref) + 1
t_ref = np.linspace(0, T_ref, num_ref)
F_ref = np.zeros((n_dof, num_ref))

selected_dofs = [0, 4, 9, 14, 19]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, (case_name, v1_0_case) in zip(axes, input_velocity_cases.items()):

    # IC for this energy / velocity case
    x0_case, xt0_case = make_left_velocity_ic(
        n_dof=n_dof,
        x1_0=x1_0,
        v1_0=v1_0_case
    )

    # Newmark reference
    u_ref, v_ref, a_ref = newmark_beta(
        M_total, C_total, K_total, F_ref,
        dt=dt_ref,
        n_steps=num_ref,
        n_dof=n_dof,
        x0=x0_case[0],
        xt0=xt0_case[0]
    )

    for i in selected_dofs:
        ax.plot(t_ref, u_ref[i, :], lw=1.2, label=f'DOF {i+1}')

    ax.set_title(f'{case_name.capitalize()} energy input: $v_1(0)$ = {v1_0_case}')
    ax.set_ylabel('Displacement')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time (s)')
axes[0].legend(ncol=5, fontsize=8)

plt.tight_layout()
plt.show()

## 4) Multi-impact PINN driver


In [ ]:
def run_free_free_pinn_multiimpact(
    t_end_target,
    T_segment,
    n_t_segment,
    nIter_per_segment,
    tau_contact=1e-3,
    x0_init=None,
    xt0_init=None,
    case_name='case',
):
    lb = np.array([0.0])
    ub = np.array([float(T_segment)])

    if x0_init is None or xt0_init is None:
        raise ValueError('x0_init and xt0_init must be provided for each case.')

    x0_cur = x0_init.copy()
    xt0_cur = xt0_init.copy()
    y0_cur = y0.copy()
    yt0_cur = yt0.copy()

    phi_cur = np.array([[0.0]])
    t_global = 0.0

    t_hist, x_hist, xt_hist, y_hist, yt_hist = [], [], [], [], []
    E_hist = []
    impact_log = []
    segment_train_times = []
    n_impacts_found = 0
    t_train_total_start = time.perf_counter()

    def segment_total_energy(x_seg, xt_seg, yt_seg):
        Ek_x = 0.5 * m_x * np.sum(xt_seg**2, axis=1)
        Ek_y = 0.5 * m_y * np.sum(yt_seg**2) * np.ones(x_seg.shape[0])
        spring_rel = x_seg[:, 1:] - x_seg[:, :-1]
        Ep_s = 0.5 * k * np.sum(spring_rel**2, axis=1)
        return Ek_x + Ek_y + Ep_s

    seg_id = 0
    while t_global < float(t_end_target):
        seg_id += 1
        t_seg = np.linspace(0.0, float(T_segment), int(n_t_segment)).reshape(-1, 1)

        seg_train_start = time.perf_counter()
        model = PIPNNs(
            lb, ub,
            np.array([[0.0]]), t_seg,
            x0_cur, xt0_cur,
            y0_cur, yt0_cur,
            M_total, K_total, D, n_dof,
            phi_cur, phi1, phi2,
            layers,
            hyp_ini_weight_loss,
            C=C_total,
            optimizer_LB=optimizer_LB_value,
        )
        model.train(nIter=int(nIter_per_segment), optimizer_LB=optimizer_LB_value)
        seg_train_time = time.perf_counter() - seg_train_start
        segment_train_times.append(seg_train_time)

        t_impacts, hit = find_impact_times(model, y0_cur, yt0_cur, D, float(T_segment))

        if np.any(hit):
            j = int(np.argmin(t_impacts))
            t_event = float(t_impacts[j])
            has_impact = np.isfinite(t_event) and (t_event > 0.0) and (t_event < float(T_segment))
        else:
            j = None
            t_event = float(T_segment)
            has_impact = False

        if not np.isfinite(t_event) or (t_event <= 0.0):
            print(f'[{case_name}] [stop] invalid segment event time at segment {seg_id}: {t_event}')
            break

        # trim final segment to hit t_end_target exactly
        t_remaining = float(t_end_target) - t_global
        t_event = min(t_event, t_remaining)

        t_local = np.linspace(0.0, t_event, int(n_t_segment) + 1).reshape(-1, 1)
        x_local, xt_local, _ = model.predict(t_local)
        y_local = (y0_cur + np.outer(t_local.flatten(), yt0_cur))
        E_local = segment_total_energy(x_local, xt_local, yt0_cur)

        t_global_local = t_global + t_local.flatten()
        yt_local = np.tile(yt0_cur, (len(t_local), 1))
        if len(t_hist) > 0:
            t_global_local = t_global_local[1:]
            x_local = x_local[1:, :]
            xt_local = xt_local[1:, :]
            y_local = y_local[1:, :]
            yt_local = yt_local[1:, :]
            E_local = E_local[1:]

        t_hist.append(t_global_local)
        x_hist.append(x_local)
        xt_hist.append(xt_local)
        y_hist.append(y_local)
        yt_hist.append(yt_local)
        E_hist.append(E_local)

        x_hit, xt_hit, _ = model.predict(np.array([[t_event]], dtype=np.float32))

        can_apply_impact_jump = has_impact and (t_event < t_remaining + 1e-14)
        if can_apply_impact_jump:
            xt_minus = float(xt_hit[0, j])
            yt_minus = float(yt0_cur[j])
            V0 = np.array([[xt_minus], [yt_minus]])
            V1 = A_inv_B @ V0
            xt_plus = float(V1[0])
            yt_plus = float(V1[1])

            Jx = m_x * (xt_plus - xt_minus)
            Jmag = abs(Jx)
            Feq = Jmag / max(float(tau_contact), 1e-12)

            x0_next, xt0_next, y0_next, yt0_next = propagate_ics(
                model,
                t_event,
                x_hit, xt_hit,
                y0_cur, yt0_cur,
                j,
                m_x * np.ones(n_dof), m_y * np.ones(n_dof), r,
                A_inv_B,
            )

            n_impacts_found += 1
            impact_log.append({
                'impact_id': n_impacts_found,
                'segment_id': seg_id,
                'dof': j + 1,
                't_local': t_event,
                't_global': t_global + t_event,
                'xt_minus': xt_minus,
                'yt_minus': yt_minus,
                'xt_plus': xt_plus,
                'yt_plus': yt_plus,
                'impulse_x': Jx,
                'impulse_abs': Jmag,
                'force_eq': Feq,
                'segment_train_time_s': seg_train_time,
            })
            print(
                f"[{case_name}] seg {seg_id:03d}: impact #{n_impacts_found:02d} "
                f"DOF {j+1:02d}, t_global={t_global + t_event:.6f}, train={seg_train_time:.2f}s"
            )
        else:
            x0_next = x_hit.copy()
            xt0_next = xt_hit.copy()
            y0_next = y0_cur + yt0_cur * t_event
            yt0_next = yt0_cur.copy()
            print(
                f"[{case_name}] seg {seg_id:03d}: no-impact propagation to t={t_global + t_event:.6f}s, "
                f"train={seg_train_time:.2f}s"
            )

        t_global += t_event
        phi_cur = phi_cur + t_event
        x0_cur, xt0_cur, y0_cur, yt0_cur = x0_next, xt0_next, y0_next, yt0_next

    if len(t_hist) == 0:
        return None

    total_train_time_s = time.perf_counter() - t_train_total_start
    out = {
        't': np.concatenate(t_hist),
        'x': np.vstack(x_hist),
        'xt': np.vstack(xt_hist),
        'y': np.vstack(y_hist),
        'yt': np.vstack(yt_hist),
        'energy_total': np.concatenate(E_hist),
        'impacts': impact_log,
        'n_impacts': n_impacts_found,
        'n_segments': seg_id,
        'segment_train_times_s': np.asarray(segment_train_times),
        'total_train_time_s': total_train_time_s,
        't_end_target_s': float(t_end_target),
    }
    return out


## 5) Run 3 input velocity/energy cases and simulate the first 20 seconds


In [ ]:
batch_results = {}
for case_name, v_in in input_velocity_cases.items():
    x0_case, xt0_case = make_left_velocity_ic(n_dof=n_dof, x1_0=0.0, v1_0=v_in)
    E0_case = 0.5 * m_x * v_in**2

    print('\n' + '='*72)
    print(f"Running case: {case_name} | v1_0={v_in:.3f} m/s | E0={E0_case:.6f} J")
    print(f"Target physical time: {t_end_target:.2f} s")
    print('='*72)

    res = run_free_free_pinn_multiimpact(
        t_end_target=t_end_target,
        T_segment=T_segment,
        n_t_segment=n_t_segment,
        nIter_per_segment=nIter_per_segment,
        tau_contact=tau_contact,
        x0_init=x0_case,
        xt0_init=xt0_case,
        case_name=case_name,
    )
    if res is None:
        print(f"Case {case_name} failed: no trajectories generated.")
        continue

    res['case_name'] = case_name
    res['v_input'] = v_in
    res['E0'] = E0_case
    batch_results[case_name] = res

    print(
        f"[{case_name}] impacts={res['n_impacts']}, segments={res['n_segments']}, "
        f"t_end={res['t'][-1]:.3f}s, train_total={res['total_train_time_s']:.2f}s"
    )

if len(batch_results) == 0:
    raise RuntimeError('No case completed successfully.')

## 6) Plot diagnostics


In [ ]:
#plot_case = 'medium' if 'medium' in batch_results else list(batch_results.keys())[0]
#results = batch_results[plot_case]
#print(f"\nPlotting case: {plot_case}")

case_names = ['low', 'medium', 'high']
selected_results = {
    'low': results_low,
    'medium': results_medium,
    'high': results_high,
}

fig, axes = plt.subplots(5, 3, figsize=(16, 13), sharex='col')

for col, case_name in enumerate(case_names):

    results = selected_results[case_name]

    t = results['t']
    x = results['x']
    xt = results['xt']
    y = results['y']
    yt = results['yt']
    E = results['energy_total']
    impacts = results['impacts']

    t_imp_global = np.array([it['t_global'] for it in impacts]) if len(impacts) else np.array([])
    F_imp = np.array([it['force_eq'] for it in impacts]) if len(impacts) else np.array([])

    rel = x - y

    # -------- (1) relative displacement --------
    ax = axes[0, col]
    for i in range(n_dof):
        ax.plot(t, rel[:, i], lw=0.8, alpha=0.75)

    ax.axhline(D, color='r', ls='--', lw=1.0)
    ax.axhline(-D, color='r', ls='--', lw=1.0)

    if len(t_imp_global):
        for tt in t_imp_global:
            ax.axvline(tt, color='k', lw=0.6, ls=':', alpha=0.25)

    ax.set_title(f'{case_name.capitalize()}')
    ax.set_ylabel(r'$x_i - y_i$')
    ax.grid(alpha=0.25)

    # -------- (2) velocity x --------
    ax = axes[1, col]
    for i in range(n_dof):
        ax.plot(t, xt[:, i], lw=0.8, alpha=0.75)

    if len(t_imp_global):
        for tt in t_imp_global:
            ax.axvline(tt, color='k', lw=0.6, ls=':', alpha=0.25)

    ax.set_ylabel(r'$\dot{x}_i$')
    ax.grid(alpha=0.25)

    # -------- (3) velocity y --------
    ax = axes[2, col]
    for i in range(n_dof):
        ax.plot(t, yt[:, i], lw=0.8, alpha=0.75)

    if len(t_imp_global):
        for tt in t_imp_global:
            ax.axvline(tt, color='k', lw=0.6, ls=':', alpha=0.25)

    ax.set_ylabel(r'$\dot{y}_i$')
    ax.grid(alpha=0.25)

    # -------- (4) impulse force --------
    ax = axes[3, col]
    if len(t_imp_global):
        ax.stem(
            t_imp_global,
            F_imp,
            basefmt=' ',
            linefmt='C3-',
            markerfmt='C3o'
        )

    ax.set_ylabel('F_eq')
    ax.grid(alpha=0.25)

    # -------- (5) energy --------
    ax = axes[4, col]
    ax.plot(t, E, lw=1.3, color='C2')

    if len(t_imp_global):
        for tt in t_imp_global:
            ax.axvline(tt, color='k', lw=0.6, ls=':', alpha=0.2)

    ax.set_ylabel('Energy')
    ax.set_xlabel('Time (s)')
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 7) Dispersion Curves (Ongoing)

In [ ]:
def compute_dispersion_map_from_response(x_tn, dt, d=1.0):
    """
    x_tn: response with shape (n_time, n_dof)
    returns normalized spectrum S(omega, k) for omega>=0 and k in [0, pi]
    """
    x_tn = np.asarray(x_tn, dtype=float)
    n_t, n_x = x_tn.shape

    # remove temporal mean per dof and apply simple Hann windows
    x0 = x_tn - np.mean(x_tn, axis=0, keepdims=True)
    wt = np.hanning(n_t)[:, None]
    wx = np.hanning(n_x)[None, :]
    xw = x0 * wt * wx

    # full 2D FFT -> then keep positive omega and nonnegative k
    F = np.fft.fft2(xw)
    S_full = np.abs(F)

    omega_raw = 2.0 * np.pi * np.fft.fftfreq(n_t, d=dt)
    k_raw = 2.0 * np.pi * np.fft.fftfreq(n_x, d=d)

    omega_mask = omega_raw >= 0.0
    k_mask = (k_raw >= 0.0) & (k_raw <= np.pi / d + 1e-12)

    omega = omega_raw[omega_mask]
    k = k_raw[k_mask]
    S = S_full[np.ix_(omega_mask, k_mask)]

    if np.max(S) > 0:
        S = S / np.max(S)

    # S returned as (n_omega, n_k)
    return k, omega, S


def plot_dispersion_for_batch(batch_results, dt, d=1.0, omega_max=None):
    case_names = list(batch_results.keys())
    n_case = len(case_names)

    fig, axes = plt.subplots(1, n_case, figsize=(5 * n_case, 4), sharey=True)
    if n_case == 1:
        axes = [axes]

    for ax, case_name in zip(axes, case_names):
        res = batch_results[case_name]
        x_tn = res['x']
        print(x_tn.shape)
        k, omega, S = compute_dispersion_map_from_response(x_tn, dt=dt, d=d)

        kpi = k / (np.pi / d)
        pcm = ax.pcolormesh(kpi, omega, S, shading='auto', cmap='magma')
        ax.set_title(f"{case_name} (v0={res['v_input']:.2f} m/s)")
        ax.set_xlabel('k / π')
        ax.set_xlim(0.0, 1.0)
        if omega_max is not None:
            ax.set_ylim(0.0, omega_max)
        ax.grid(alpha=0.2)

        fig.colorbar(pcm, ax=ax, fraction=0.046, pad=0.04, label='Normalized |FFT|')

    axes[0].set_ylabel('ω (rad/s)')
    fig.suptitle('Dispersion maps from simulated responses (3 velocity cases)', y=1.02)
    fig.tight_layout()
    return fig, axes


# estimate dt directly from simulated time grid
first_case = next(iter(batch_results.values()))
dt_est = float(np.mean(np.diff(first_case['t'][: min(len(first_case['t']), 200)])))
fig, axes = plot_dispersion_for_batch(batch_results, dt=dt_est, d=1.0, omega_max=2.0)
plt.show()


## 8) Save all case results


In [ ]:
save_dir = 'Result_free_free'
os.makedirs(save_dir, exist_ok=True)

# save each case independently (NPZ + MAT when scipy is available)
try:
    from scipy.io import savemat
    has_savemat = True
except Exception:
    has_savemat = False

summary_rows = []
for case_name, res in batch_results.items():
    stem = f"pinn_free_free_20dof_{case_name}"
    npz_path = os.path.join(save_dir, stem + '.npz')

    np.savez(
        npz_path,
        t=res['t'],
        x=res['x'],
        xt=res['xt'],
        y=res['y'],
        yt=res['yt'],
        energy_total=res['energy_total'],
        impacts=np.array(res['impacts'], dtype=object),
        segment_train_times_s=res['segment_train_times_s'],
        total_train_time_s=res['total_train_time_s'],
        v_input=res['v_input'],
        E0=res['E0'],
    )

    if has_savemat:
        mat_path = os.path.join(save_dir, stem + '.mat')
        savemat(mat_path, {
            't_total': res['t'],
            'x_PINN_total': res['x'],
            'xt_PINN_total': res['xt'],
            'y_total': res['y'],
            'yt_total': res['yt'],
            'energy_total': res['energy_total'],
            'v_input': np.array([[res['v_input']]]),
            'E0': np.array([[res['E0']]]),
            'n_impacts': np.array([[res['n_impacts']]]),
            'n_segments': np.array([[res['n_segments']]]),
            'train_time_total_s': np.array([[res['total_train_time_s']]]),
        })

    summary_rows.append((case_name, res['v_input'], res['E0'], res['n_impacts'], res['n_segments'], res['t'][-1], res['total_train_time_s']))

# save global summary table
summary_path = os.path.join(save_dir, 'batch_summary.csv')
with open(summary_path, 'w') as f:
    f.write('case,v_input,E0,n_impacts,n_segments,t_end_s,total_train_time_s\n')
    for row in summary_rows:
        f.write(','.join([str(v) for v in row]) + '\n')

print('Saved all cases to:', save_dir)
print('MAT export available:', has_savemat)
for row in summary_rows:
    print(f"case={row[0]:>6s} | v={row[1]:>6.2f} | E0={row[2]:>9.4f} | impacts={row[3]:>3d} | segments={row[4]:>3d} | t_end={row[5]:>6.2f}s | train={row[6]:>8.2f}s")

